# Implement and Test Text Chunking

This notebook implements an initial word-based text chunking strategy for the cleaned research-paper dataset. It experiments with chunk size and overlap while preserving page-level metadata, then reviews sample chunks to confirm that technical context is retained.

In [16]:
import json
from pathlib import Path
import pandas as pd

data_path = Path("../data/processed/cleaned_papers.json")
with data_path.open("r", encoding="utf-8") as file:
    records = json.load(file)

print("Dataset loaded successfully.")
print("Input records:", len(records))
print("Input path:", data_path)

Dataset loaded successfully.
Input records: 218
Input path: ..\data\processed\cleaned_papers.json


In [17]:
required_fields = {
    "paper_id", "title", "category", "pdf_url",
    "page_number", "cleaned_text"
}
metadata_string_fields = {"paper_id", "title", "category", "pdf_url"}

assert isinstance(records, list), "records must be a list"
assert records, "records must not be empty"
assert all(isinstance(record, dict) for record in records), "Every record must be a dictionary"

missing_fields_by_record = [
    required_fields - set(record)
    for record in records
]
records_with_missing_fields = sum(bool(fields) for fields in missing_fields_by_record)
invalid_metadata_records = [
    index for index, record in enumerate(records)
    if any(not isinstance(record.get(field), str) or not record.get(field, "").strip() for field in metadata_string_fields)
    or not isinstance(record.get("page_number"), int)
    or isinstance(record.get("page_number"), bool)
    or record.get("page_number", 0) <= 0
]
invalid_text_records = [
    index for index, record in enumerate(records)
    if not isinstance(record.get("cleaned_text"), str)
    or not record.get("cleaned_text", "").strip()
]

In [18]:
source_page_keys = [
    (record.get("paper_id"), record.get("page_number"))
    for record in records
]
duplicate_source_pages = len(source_page_keys) - len(set(source_page_keys))

print("Records with missing fields:", records_with_missing_fields)
print("Records with invalid metadata:", len(invalid_metadata_records))
print("Records with invalid or empty text:", len(invalid_text_records))
print("Duplicate paper-page records:", duplicate_source_pages)

assert records_with_missing_fields == 0
assert not invalid_metadata_records
assert not invalid_text_records
assert duplicate_source_pages == 0

pd.DataFrame(records).head()

Records with missing fields: 0
Records with invalid metadata: 0
Records with invalid or empty text: 0
Duplicate paper-page records: 0


,paper_id,title,category,pdf_url,page_number,cleaned_text
0,2005.11401,Retrieval-Augmented Generation for Knowledge-I...,RAG,https://arxiv.org/pdf/2005.11401.pdf,1,Retrieval-Augmented Generation for Knowledge-I...
1,2005.11401,Retrieval-Augmented Generation for Knowledge-I...,RAG,https://arxiv.org/pdf/2005.11401.pdf,2,The Divine Comedy (x) q Query Encoder q(x) MIP...
2,2005.11401,Retrieval-Augmented Generation for Knowledge-I...,RAG,https://arxiv.org/pdf/2005.11401.pdf,3,byθ that generates a current token based on a ...
3,2005.11401,Retrieval-Augmented Generation for Knowledge-I...,RAG,https://arxiv.org/pdf/2005.11401.pdf,4,minimize the negative marginal log-likelihood ...
4,2005.11401,Retrieval-Augmented Generation for Knowledge-I...,RAG,https://arxiv.org/pdf/2005.11401.pdf,5,MSMARCO as an open-domain abstractive QA task....


## Chunking Strategy

The initial strategy uses word boundaries rather than character boundaries. Each chunk has a fixed maximum number of words, and adjacent chunks share an overlapping window. Splitting on words avoids cutting technical terms in the middle and makes chunk size and overlap easy to compare.

In [19]:
import textwrap


def chunk_text(text, chunk_size=400, overlap=80):
    """Split text into overlapping word-based chunks with validation."""
    if not isinstance(text, str):
        raise TypeError("text must be a string")
    if not isinstance(chunk_size, int) or isinstance(chunk_size, bool) or chunk_size <= 0:
        raise ValueError("chunk_size must be a positive integer")
    if not isinstance(overlap, int) or isinstance(overlap, bool) or overlap < 0 or overlap >= chunk_size:
        raise ValueError("overlap must be a non-negative integer smaller than chunk_size")

    words = text.split()
    if not words:
        return []

    step = chunk_size - overlap
    chunks = []

    for start in range(0, len(words), step):
        chunk_words = words[start:start + chunk_size]
        if not chunk_words:
            break
        chunks.append({
            "text": " ".join(chunk_words),
            "start_word": start,
            "end_word": start + len(chunk_words) - 1,
            "word_count": len(chunk_words)
        })
        if start + chunk_size >= len(words):
            break

    assert chunks[0]["start_word"] == 0
    assert chunks[-1]["end_word"] == len(words) - 1
    for previous, current in zip(chunks, chunks[1:]):
        assert current["start_word"] - previous["start_word"] == step
        assert current["start_word"] <= previous["end_word"] + 1

    return chunks

In [20]:
def chunk_records(records, chunk_size=400, overlap=80):
    """Chunk every page while carrying its source metadata forward."""
    chunked_records = []
    metadata_fields = [
        "paper_id", "title", "category", "pdf_url", "page_number"
    ]

    for record in records:
        page_chunks = chunk_text(record["cleaned_text"], chunk_size, overlap)
        for chunk_index, chunk in enumerate(page_chunks):
            chunked_records.append({
                **{field: record[field] for field in metadata_fields},
                "chunk_id": f"{record['paper_id']}-page-{record['page_number']}-chunk-{chunk_index}",
                "chunk_index": chunk_index,
                "chunk_size": chunk_size,
                "overlap": overlap,
                **chunk
            })

    return chunked_records


sample_chunks = chunk_text(records[0]["cleaned_text"], chunk_size=40, overlap=10)
sample_chunk = sample_chunks[0]
print("Sample chunk created")
print("-" * 80)
print("Word range:", f"{sample_chunk['start_word']} to {sample_chunk['end_word']}")
print("Word count:", sample_chunk["word_count"])
print("Text:")
print(textwrap.fill(sample_chunk["text"], width=100))

Sample chunk created
--------------------------------------------------------------------------------
Word range: 0 to 39
Word count: 40
Text:
Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks Patrick Lewis†‡, Ethan Perez⋆,
Aleksandra Piktus†, Fabio Petroni†, Vladimir Karpukhin†, Naman Goyal†, Heinrich Küttler†, Mike
Lewis†, Wen-tau Yih†, Tim Rocktäschel†‡, Sebastian Riedel†‡, Douwe Kiela† †Facebook AI
Research;‡University College London;⋆New York University; plewis@fb.com Abstract Large


In [21]:
configurations = [
    {"chunk_size": 200, "overlap": 40},
    {"chunk_size": 400, "overlap": 80},
    {"chunk_size": 800, "overlap": 160}
]

experiment_rows = []
experiment_outputs = {}

for config in configurations:
    key = (config["chunk_size"], config["overlap"])
    chunks = chunk_records(records, **config)
    experiment_outputs[key] = chunks
    word_counts = [chunk["word_count"] for chunk in chunks]
    character_counts = [len(chunk["text"]) for chunk in chunks]
    experiment_rows.append({
        "Chunk Size": config["chunk_size"],
        "Overlap": config["overlap"],
        "Total Chunks": len(chunks),
        "Average Words": round(sum(word_counts) / len(word_counts), 1),
        "Minimum Words": min(word_counts),
        "Maximum Words": max(word_counts),
        "Average Characters": round(sum(character_counts) / len(character_counts), 1)
    })

experiment_summary = pd.DataFrame(experiment_rows)
experiment_summary

,Chunk Size,Overlap,Total Chunks,Average Words,Minimum Words,Maximum Words,Average Characters
0,200,40,809,178.1,41,200,1178.5
1,400,80,426,321.8,79,400,2130.4
2,800,160,228,535.3,79,800,3548.4


## Recommended Configuration and Metadata Review

The 400-word chunks with 80-word overlap provide a middle ground between retrieval precision and enough surrounding technical context. The following cells create that candidate dataset and inspect representative chunks from multiple papers.

In [22]:
recommended_chunk_size = 400
recommended_overlap = 80
chunked_records = chunk_records(
    records,
    chunk_size=recommended_chunk_size,
    overlap=recommended_overlap
)

chunked_df = pd.DataFrame(chunked_records)
metadata_columns = [
    "paper_id", "title", "category", "pdf_url", "page_number",
    "chunk_id", "chunk_index", "chunk_size", "overlap",
    "text", "start_word", "end_word", "word_count"
]

print("Recommended chunks:", len(chunked_df))
print("Missing metadata values:", chunked_df[metadata_columns].isna().sum().sum())
assert list(chunked_df.columns) == metadata_columns
assert chunked_df[metadata_columns].isna().sum().sum() == 0
assert chunked_df["chunk_id"].is_unique
assert (chunked_df["chunk_size"] == recommended_chunk_size).all()
assert (chunked_df["overlap"] == recommended_overlap).all()
assert (chunked_df["word_count"] == chunked_df["text"].str.split().str.len()).all()
assert (chunked_df["word_count"] <= chunked_df["chunk_size"]).all()
assert chunked_df.groupby(["paper_id", "page_number"])["chunk_index"].min().eq(0).all()

chunked_df[metadata_columns].head(3)

Recommended chunks: 426
Missing metadata values: 0


,paper_id,title,category,pdf_url,page_number,chunk_id,chunk_index,chunk_size,overlap,text,start_word,end_word,word_count
0,2005.11401,Retrieval-Augmented Generation for Knowledge-I...,RAG,https://arxiv.org/pdf/2005.11401.pdf,1,2005.11401-page-1-chunk-0,0,400,80,Retrieval-Augmented Generation for Knowledge-I...,0,392,393
1,2005.11401,Retrieval-Augmented Generation for Knowledge-I...,RAG,https://arxiv.org/pdf/2005.11401.pdf,2,2005.11401-page-2-chunk-0,0,400,80,The Divine Comedy (x) q Query Encoder q(x) MIP...,0,399,400
2,2005.11401,Retrieval-Augmented Generation for Knowledge-I...,RAG,https://arxiv.org/pdf/2005.11401.pdf,2,2005.11401-page-2-chunk-1,1,400,80,whereby both the generator and retriever are j...,320,640,321


In [23]:
sample_papers = chunked_df["paper_id"].drop_duplicates().head(2)
sample_chunks_df = chunked_df[chunked_df["paper_id"].isin(sample_papers)].groupby("paper_id").head(2)

for _, row in sample_chunks_df.iterrows():
    print("=" * 80)
    print("TITLE:", row["title"])
    print("PAGE:", row["page_number"], "CHUNK:", row["chunk_index"])
    print("WORD RANGE:", f"{row['start_word']} to {row['end_word']}")
    print("PREVIEW:")
    print(row["text"][:1200])
    print()

TITLE: Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks
PAGE: 1 CHUNK: 0
WORD RANGE: 0 to 392
PREVIEW:
Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks Patrick Lewis†‡, Ethan Perez⋆, Aleksandra Piktus†, Fabio Petroni†, Vladimir Karpukhin†, Naman Goyal†, Heinrich Küttler†, Mike Lewis†, Wen-tau Yih†, Tim Rocktäschel†‡, Sebastian Riedel†‡, Douwe Kiela† †Facebook AI Research;‡University College London;⋆New York University; plewis@fb.com Abstract Large pre-trained language models have been shown to store factual knowledge in their parameters, and achieve state-of-the-art results when fine-tuned on downstream NLP tasks. However, their ability to access and precisely manipulate knowledge is still limited, and hence on knowledge-intensive tasks, their performance lags behind task-specific architectures. Additionally, providing provenance for their decisions and updating their world knowledge remain open research problems. Pretrained models with a differentiable 

## Technical-Context Retention Check

This lightweight check selects distinctive terms from each source page and measures whether those terms are represented in the chunks generated for that page. It is not a semantic evaluation, but it helps identify obvious loss of technical vocabulary during chunking.

In [24]:
from collections import defaultdict

PHRASE_WINDOW = 25


def group_chunks_by_page(chunks):
    """Group chunk dictionaries by (paper_id, page_number), keeping chunk order."""
    grouped = defaultdict(list)
    for chunk in chunks:
        grouped[(chunk["paper_id"], chunk["page_number"])].append(chunk)
    return grouped

In [25]:
def check_chunks(records, chunks, window=PHRASE_WINDOW):
    """Compare chunks to the original pages and count phrases that stay intact."""
    grouped = group_chunks_by_page(chunks)
    rows = []

    for record in records:
        words = record["cleaned_text"].split()
        page_chunks = grouped.get((record["paper_id"], record["page_number"]), [])

        text_matches = bool(page_chunks) and all(
            chunk["text"].split() == words[chunk["start_word"]:chunk["end_word"] + 1]
            for chunk in page_chunks
        )

        no_gaps = bool(page_chunks) and (
            page_chunks[0]["start_word"] == 0
            and page_chunks[-1]["end_word"] == len(words) - 1
            and all(
                nxt["start_word"] <= prev["end_word"] + 1
                for prev, nxt in zip(page_chunks, page_chunks[1:])
            )
        )

        total_phrases = max(len(words) - window + 1, 0)
        intact_phrases = sum(
            any(
                chunk["start_word"] <= i and i + window - 1 <= chunk["end_word"]
                for chunk in page_chunks
            )
            for i in range(total_phrases)
        )

        rows.append({
            "paper_id": record["paper_id"],
            "page_number": record["page_number"],
            "text_matches": text_matches,
            "no_gaps": no_gaps,
            "total_phrases": total_phrases,
            "intact_phrases": intact_phrases,
        })

    return pd.DataFrame(rows)

In [26]:
comparison_configs = [
    {"chunk_size": 400, "overlap": 0},
    {"chunk_size": 200, "overlap": 40},
    {"chunk_size": 400, "overlap": 80},
    {"chunk_size": 800, "overlap": 160},
]

comparison_rows = []
for config in comparison_configs:
    key = (config["chunk_size"], config["overlap"])
    chunks = experiment_outputs[key] if key in experiment_outputs else chunk_records(records, **config)
    result = check_chunks(records, chunks)
    total = result["total_phrases"].sum()
    comparison_rows.append({
        "Chunk Size": config["chunk_size"],
        "Overlap": config["overlap"],
        "Pages": len(result),
        "Pages With Matching Text": int(result["text_matches"].sum()),
        "Pages With No Gaps": int(result["no_gaps"].sum()),
        "Phrase Retention": round(result["intact_phrases"].sum() / total, 4) if total else 1.0,
    })

retention_summary = pd.DataFrame(comparison_rows)

recommended_row = retention_summary[
    (retention_summary["Chunk Size"] == recommended_chunk_size)
    & (retention_summary["Overlap"] == recommended_overlap)
].iloc[0]
assert recommended_row["Pages With Matching Text"] == recommended_row["Pages"]
assert recommended_row["Pages With No Gaps"] == recommended_row["Pages"]
assert recommended_row["Phrase Retention"] == 1.0

retention_summary

,Chunk Size,Overlap,Pages,Pages With Matching Text,Pages With No Gaps,Phrase Retention
0,400,0,218,218,218,0.9601
1,200,40,218,218,218,1.0000
2,400,80,218,218,218,1.0000
3,800,160,218,218,218,1.0000
